# Vehicle Price Prediction — Model Training (Team 1)

Train four regression models with **GridSearchCV**, evaluate with **RMSE / MAE / R²**,
inspect **residual** and **QQ** plots, and select the best model.
All results are reproducible from `data/processed/` (the committed train/test split).

In [ ]:
import pandas as pd, numpy as np, json, os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize']=(12,5)

In [ ]:
# Load the preprocessed split produced by DataPreprocessing.ipynb
DATA = '../data/processed/'
X_train = pd.read_csv(DATA+'X_train.csv'); X_test = pd.read_csv(DATA+'X_test.csv')
y_train = pd.read_csv(DATA+'y_train.csv').values.ravel()
y_test  = pd.read_csv(DATA+'y_test.csv').values.ravel()
print('Train:', X_train.shape, ' Test:', X_test.shape)

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    pred = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, pred)
    mae  = mean_absolute_error(y_test, pred)
    r2   = r2_score(y_test, pred)
    print('='*50); print(f'Model: {name}')
    print(f'  RMSE: ${rmse:,.2f}\n  MAE:  ${mae:,.2f}\n  R²:   {r2:.4f}')
    return {'name': name, 'rmse': float(rmse), 'mae': float(mae), 'r2': float(r2), 'pred': pred}

def plot_residuals(name, y_test, pred, save_path=None):
    residuals = y_test - pred
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.scatter(pred, residuals, alpha=0.2, s=5, color='steelblue'); ax1.axhline(0, color='red', lw=1.5)
    ax1.set_xlabel('Predicted Price ($)'); ax1.set_ylabel('Residuals ($)'); ax1.set_title(f'{name} — Residuals Plot')
    stats.probplot(residuals, dist='norm', plot=ax2); ax2.set_title(f'{name} — QQ Plot')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

## 1. Ridge Regression

In [ ]:
ridge_gs = GridSearchCV(Ridge(), {'alpha': [0.1, 1.0, 10.0, 100.0]},
                        scoring='neg_root_mean_squared_error', cv=3, n_jobs=-1)
ridge_gs.fit(X_train, y_train)
print('Best params:', ridge_gs.best_params_)
results_ridge = evaluate_model('Ridge Regression', ridge_gs.best_estimator_, X_test, y_test)
plot_residuals('Ridge Regression', y_test, results_ridge['pred'], '../reports/model1_ridge_plots.png')

## 2. Random Forest Regressor

In [ ]:
rf_gs = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                     {'n_estimators': [50, 100], 'max_depth': [10, 20], 'criterion': ['squared_error']},
                     scoring='neg_root_mean_squared_error', cv=3, n_jobs=-1)
rf_gs.fit(X_train, y_train)
print('Best params:', rf_gs.best_params_)
results_rf = evaluate_model('Random Forest', rf_gs.best_estimator_, X_test, y_test)
plot_residuals('Random Forest', y_test, results_rf['pred'], '../reports/model2_rf_plots.png')

## 3. Gradient Boosting Regressor

In [ ]:
gb_gs = GridSearchCV(GradientBoostingRegressor(random_state=42),
                     {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]},
                     scoring='neg_root_mean_squared_error', cv=3, n_jobs=-1)
gb_gs.fit(X_train, y_train)
print('Best params:', gb_gs.best_params_)
results_gb = evaluate_model('Gradient Boosting', gb_gs.best_estimator_, X_test, y_test)
plot_residuals('Gradient Boosting', y_test, results_gb['pred'], '../reports/model3_gb_plots.png')

## 4. K-Nearest Neighbors Regressor

In [ ]:
knn_gs = GridSearchCV(KNeighborsRegressor(),
                      {'n_neighbors': [10, 30, 50], 'weights': ['uniform', 'distance'], 'metric': ['cosine']},
                      scoring='neg_root_mean_squared_error', cv=3, n_jobs=-1)
knn_gs.fit(X_train, y_train)
print('Best params:', knn_gs.best_params_)
results_knn = evaluate_model('K-Nearest Neighbors', knn_gs.best_estimator_, X_test, y_test)
plot_residuals('K-Nearest Neighbors', y_test, results_knn['pred'], '../reports/model4_knn_plots.png')

## Models Comparison & Best Model

In [ ]:
all_results = [results_ridge, results_rf, results_gb, results_knn]
summary = pd.DataFrame([{k:v for k,v in r.items() if k!='pred'} for r in all_results]).sort_values('rmse')
print(summary.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric, label, col in zip(axes, ['rmse','mae','r2'], ['RMSE ($)','MAE ($)','R²'], ['steelblue','coral','teal']):
    ax.barh(summary['name'], summary[metric], color=col, alpha=0.8)
    ax.set_xlabel(label); ax.set_title(f'Model Comparison — {label}'); ax.invert_yaxis()
plt.tight_layout(); plt.savefig('../reports/model_comparison.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
# Best model = lowest RMSE. Save it as the production artifact.
import joblib
model_map = {'Ridge Regression': ridge_gs.best_estimator_, 'Random Forest': rf_gs.best_estimator_,
             'Gradient Boosting': gb_gs.best_estimator_, 'K-Nearest Neighbors': knn_gs.best_estimator_}
best_name = summary.iloc[0]['name']
print('Best model:', best_name)
os.makedirs('../models/best', exist_ok=True)
joblib.dump(model_map[best_name], '../models/best/best_model.joblib')

# persist metrics for the report
for r in all_results:
    json.dump({k:v for k,v in r.items() if k!='pred'},
              open(f"../data/processed/{r['name'].split()[0].lower()}_results.json", 'w'), indent=2)
print('Saved best model -> models/best/best_model.joblib')